In [1]:
# mike babb
# created: 2026 08 23
# updated: 2026 09 23
# find five words with 25 different letters
# example: ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']

In [2]:
# standard
import math
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

**Multiprocessing note:** the two heavy combinatorial loops below (building level 2, and the main level-4/level-5 search) now run across a `multiprocessing.Pool` instead of a single Python loop. The worker functions live in `mp_worker.py`, next to this notebook -- they can't be defined inline in the notebook because worker processes need to be able to `import` and pickle them.

In [5]:
# multiprocessing
import multiprocessing as mp
from mp_worker import (
    _init_worker, process_chunk,
    _init_l2_worker, process_l2_chunk,
)

# mp_worker.py must live next to this notebook (or on sys.path) --
# the worker functions have to be importable from a real module so
# they can be pickled and sent to worker processes. This matters most
# on macOS/Windows, where multiprocessing defaults to the 'spawn'
# start method; on Linux ('fork') it isn't strictly required, but it
# keeps the notebook portable either way.

# leave one core free for the OS / the notebook kernel itself
n_workers = max(1, mp.cpu_count() - 1)
print(f'using {n_workers} worker processes')

using 19 worker processes


In [6]:
# CREATE A TEST VARIABLE
use_test = False

# LOAD DATA

In [7]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [8]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## DEMONSTRATE BITWISE OPERATIONS

In [9]:
vibex = byte_encode_words('vibex')
glyph = byte_encode_words('glyph')
muntz = byte_encode_words('muntz')
dwarf = byte_encode_words('dwarf')
jocks = byte_encode_words('jocks')
cramp = byte_encode_words('cramp')

In [10]:
# this is equal to zero - no letters reused
(vibex | glyph | muntz | dwarf) & jocks 

0

In [11]:
# this is not equal to zero because letters are reused
((vibex | glyph) | muntz | dwarf) & cramp

167937

In [12]:
# order of operations for bitwise operations
(vibex | glyph) & (muntz | dwarf) 

0

In [13]:
vibex | glyph | muntz | dwarf | jocks

67043327

In [14]:
# same output as above
byte_encode_words('vibexglyphmuntzdwarfjocks')

67043327

# BUILD LEVEL 2 BY COMBINING TWO BYTE ENCODED WORDS

In [15]:
# how many outer-index values (i.e. how many i's in
# combinations(word_byte_list, 2)) each worker task covers
l2_rows_per_chunk = 50

n_words = len(word_byte_list)
l2_index_ranges = [
    (i, min(i + l2_rows_per_chunk, n_words))
    for i in range(0, n_words, l2_rows_per_chunk)
]
print(f'{n_words} words split into {len(l2_index_ranges)} chunks')

l2_results = []
with mp.Pool(
    processes=n_workers,
    initializer=_init_l2_worker,
    initargs=(word_byte_list,),
) as pool:
    for i, chunk_result in enumerate(
        pool.imap_unordered(process_l2_chunk, l2_index_ranges), start=1
    ):
        if chunk_result.shape[0] > 0:
            l2_results.append(chunk_result)
        if i % 50 == 0 or i == len(l2_index_ranges):
            print(f'{i}/{len(l2_index_ranges)} chunks complete')

l2_list = (
    np.vstack(l2_results) if l2_results else np.empty(shape=(0, 3), dtype=np.int32)
)
l2_df = pd.DataFrame(data=l2_list, columns=['w1b', 'w2b', 'l2'])
l2_df.shape

5977 words split into 120 chunks
50/120 chunks complete
100/120 chunks complete
120/120 chunks complete


(3213696, 3)

# BUILD LEVELS 4 AND 5 BY COMBINING TWO ITEMS FROM THE L2 LIST  
# COMPARE THAT WITH THE WORD BYTE ARRAY ONE MORE TIME

In [16]:
# get words from bytes
l2_df['w1'] = l2_df['w1b'].map(word_byte_to_word_dict)
l2_df['w2'] = l2_df['w2b'].map(word_byte_to_word_dict)

In [17]:
w_l2_df = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [18]:
w_l2_all = w_l2_df['l2'].to_numpy(dtype = np.int32)

In [19]:
# let's just use the word jocks
if use_test:
    w_l2_df = w_l2_df.loc[(w_l2_df['w1'] == 'jocks') |
                                (w_l2_df['w2'] == 'jocks'), :].reset_index(drop = True)

In [20]:
w_l2_df = w_l2_df.drop(labels = ['w1', 'w2'], axis = 1)
l2_df = l2_df.drop(labels = ['w1', 'w2'], axis = 1)

In [21]:
w_l2_df.shape

(640023, 3)

In [22]:
w_l2_df['l2'].unique().shape

(640023,)

In [23]:
w_l2_all.shape

(640023,)

In [24]:
# we are going to make a lot of comparisons
print('The full set of l2 - duplicated l2:', l2_df.shape[0], l2_df.shape[0] ** 2)
print('The unique l2:', l2_df['l2'].unique().shape[0],  l2_df['l2'].unique().shape[0]** 2)
# but, we'll be clever about this and compare each item from the l2_list
# against the whole l2_list using array operations. 


The full set of l2 - duplicated l2: 3213696 10327841980416
The unique l2: 640023 409629440529


# COMPUTE THE COMBINATIONS

In [25]:
w_l2_df.shape

(640023, 3)

In [26]:
l2_df.head()

,w1b,w2b,l2
0,20491,264468,284959
1,20491,532756,553247
2,20491,788500,808991
3,20491,794644,815135
4,20491,1114388,1134879


In [27]:
w_l2_all.shape

(640023,)

In [28]:
w_l2_df.head()

,w1b,w2b,l2
0,20491,264468,284959
1,20491,532756,553247
2,20491,788500,808991
3,20491,794644,815135
4,20491,1114388,1134879


In [29]:
w_l2_df.shape

(640023, 3)

In [30]:
l2_df.shape

(3213696, 3)

In [31]:
l2_all = l2_df['l2'].to_numpy(dtype = np.int32)

In [32]:
# how many l2_df rows each worker task covers; smaller chunks give
# better load balancing (a worker that finishes an 'easy' chunk picks
# up the next one) at the cost of a bit more scheduling overhead
rows_per_chunk = 200

n_rows = l2_df.shape[0]
w1b_arr = l2_df['w1b'].to_numpy(dtype=np.int32)
w2b_arr = l2_df['w2b'].to_numpy(dtype=np.int32)
l2_col_arr = l2_df['l2'].to_numpy(dtype=np.int32)

index_ranges = [
    (i, min(i + rows_per_chunk, n_rows))
    for i in range(0, n_rows, rows_per_chunk)
]
print(f'{n_rows} l2 rows split into {len(index_ranges)} chunks')

results = []
with mp.Pool(
    processes=n_workers,
    initializer=_init_worker,
    initargs=(l2_col_arr, word_byte_array, w1b_arr, w2b_arr, l2_col_arr),
) as pool:
    for i, chunk_result in enumerate(
        pool.imap_unordered(process_chunk, index_ranges), start=1
    ):
        if chunk_result.shape[0] > 0:
            results.append(chunk_result)
        if i % 50 == 0 or i == len(index_ranges):
            print(f'{i}/{len(index_ranges)} chunks complete')

total_output = (
    np.vstack(results) if results else np.empty(shape=(0, 5), dtype=np.int32)
)
start_pos = total_output.shape[0]
print(total_output.shape)

3213696 l2 rows split into 16069 chunks
50/16069 chunks complete
100/16069 chunks complete
150/16069 chunks complete
200/16069 chunks complete
250/16069 chunks complete
300/16069 chunks complete
350/16069 chunks complete
400/16069 chunks complete
450/16069 chunks complete
500/16069 chunks complete
550/16069 chunks complete
600/16069 chunks complete
650/16069 chunks complete
700/16069 chunks complete
750/16069 chunks complete
800/16069 chunks complete
850/16069 chunks complete
900/16069 chunks complete
950/16069 chunks complete
1000/16069 chunks complete
1050/16069 chunks complete
1100/16069 chunks complete
1150/16069 chunks complete
1200/16069 chunks complete
1250/16069 chunks complete
1300/16069 chunks complete
1350/16069 chunks complete
1400/16069 chunks complete
1450/16069 chunks complete
1500/16069 chunks complete
1550/16069 chunks complete
1600/16069 chunks complete
1650/16069 chunks complete
1700/16069 chunks complete
1750/16069 chunks complete
1800/16069 chunks complete
1850/160

# CREATE AND SHAPE THE OUTPUT

In [33]:
n_workers

19

In [34]:
output = total_output[:start_pos]

In [35]:
# turn it into a dataframe
output_df = pd.DataFrame(data = output, columns = ['w1b', 'w2b', 'l2',  'l3l4', 'w5b'])
output_df.shape

(16140, 5)

In [36]:
output_df.head()

,w1b,w2b,l2,l3l4,w5b
0,2098437,16785938,18884375,42879080,5279872
1,2098437,16785938,18884375,14194856,33964096
2,2098437,16785938,18884375,39243968,8914984
3,2098437,8914984,11013421,50750034,5279872
4,2098437,8914984,11013421,22065810,33964096


# JOIN TO GET THE W3B AND THE W4B

In [37]:
l3l4_df = l2_df[['w1b', 'w2b', 'l2']].copy()
l3l4_df.columns = ['w3b', 'w4b', 'l3l4',]

In [38]:
l3l4_df.shape

(3213696, 3)

In [39]:
output_df = pd.merge(left = output_df, right = l3l4_df)

In [40]:
output_df.shape

(31314, 7)

In [41]:
output_df.head()

,w1b,w2b,l2,l3l4,w5b,w3b,w4b
0,2098437,16785938,18884375,42879080,5279872,8914984,33964096
1,2098437,16785938,18884375,14194856,33964096,8914984,5279872
2,2098437,16785938,18884375,39243968,8914984,33964096,5279872
3,2098437,8914984,11013421,50750034,5279872,16785938,33964096
4,2098437,8914984,11013421,22065810,33964096,16785938,5279872


In [42]:
# reorder...
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b']
output_df = output_df[col_names].copy()

In [43]:
# get words!
for ii in range(1, 6):
    bcn = f"w{ii}b"
    cn = f"w{ii}"
    output_df[cn] = output_df[bcn].map(word_byte_to_word_dict)

In [44]:
# count the remainder letter
lc_set = set(ascii_lowercase)
def get_remainder_letter(row):
    my_set = set()
    for cn in ['w1', 'w2', 'w3', 'w4', 'w5']:
        my_set.update(row[cn])

    return ''.join(lc_set.difference(my_set))

output_df['remaining_letter'] = output_df.apply(get_remainder_letter, axis = 1)

In [45]:
# get the word group
def get_remainder_letter(row):
    my_set = set()
    for cn in ['w1', 'w2', 'w3', 'w4', 'w5']:
        my_set.add(row[cn])

    return ' '.join(sorted(my_set))

output_df['word_group'] = output_df.apply(get_remainder_letter, axis = 1)

In [46]:
# count unique words - JUST TO VERIFY
col_names = ['w1', 'w2', 'w3', 'w4', 'w5']
output_df['n_unique_words'] = output_df[col_names].apply(lambda x: len(set(x)), axis = 1)

In [47]:
# add the words - ALSO TO VERIFY
output_df['bitwise_or'] = 0
output_df['bitwise_and'] = 0
for cn_idx in range(1, 6):
    b_cn = f"w{cn_idx}b"
    w_cn = f"w{cn_idx}"
    output_df[w_cn] = output_df[b_cn].map(word_byte_to_word_dict)
    output_df['bitwise_and'] = output_df['bitwise_and'] & output_df[b_cn]
    output_df['bitwise_or'] = output_df['bitwise_or'] | output_df[b_cn]


In [48]:
output_df.head()

,w1b,w2b,w3b,w4b,w5b,w1,w2,w3,w4,w5,remaining_letter,word_group,n_unique_words,bitwise_or,bitwise_and
0,2098437,16785938,8914984,33964096,5279872,avick,benjy,fldxt,grosz,whump,q,avick benjy fldxt grosz whump,5,67043327,0
1,2098437,16785938,8914984,5279872,33964096,avick,benjy,fldxt,whump,grosz,q,avick benjy fldxt grosz whump,5,67043327,0
2,2098437,16785938,33964096,5279872,8914984,avick,benjy,grosz,whump,fldxt,q,avick benjy fldxt grosz whump,5,67043327,0
3,2098437,8914984,16785938,33964096,5279872,avick,fldxt,benjy,grosz,whump,q,avick benjy fldxt grosz whump,5,67043327,0
4,2098437,8914984,16785938,5279872,33964096,avick,fldxt,benjy,whump,grosz,q,avick benjy fldxt grosz whump,5,67043327,0


In [49]:
output_df['word_group'].value_counts()

word_group
bumpy chawk fldxt girns vejoz    117
bungy chawk fldxt prims vejoz    114
bungy chimp fldxt vejoz warks    114
bumpy fldxt gnars vejoz whick    111
bungy fldxt ricks vejoz whamp    111
                                ... 
brock japyx seqwl vingt zhmud     30
bring fldxt psych quawk vejoz     30
brigs fldxt nymph quawk vejoz     30
braze fldxt gconv jumpy whisk     30
glack hdqrs jowpy muntz vibex     30
Name: count, Length: 538, dtype: int64

In [50]:
output_df['bitwise_or'].unique().shape

(11,)

# CREATE AND SAVE OUTPUT

In [51]:
output_df.to_excel(excel_writer='test.xlsx', index = False)

In [52]:
output_df.shape

(31314, 15)

In [53]:
output_df.head()

,w1b,w2b,w3b,w4b,w5b,w1,w2,w3,w4,w5,remaining_letter,word_group,n_unique_words,bitwise_or,bitwise_and
0,2098437,16785938,8914984,33964096,5279872,avick,benjy,fldxt,grosz,whump,q,avick benjy fldxt grosz whump,5,67043327,0
1,2098437,16785938,8914984,5279872,33964096,avick,benjy,fldxt,whump,grosz,q,avick benjy fldxt grosz whump,5,67043327,0
2,2098437,16785938,33964096,5279872,8914984,avick,benjy,grosz,whump,fldxt,q,avick benjy fldxt grosz whump,5,67043327,0
3,2098437,8914984,16785938,33964096,5279872,avick,fldxt,benjy,grosz,whump,q,avick benjy fldxt grosz whump,5,67043327,0
4,2098437,8914984,16785938,5279872,33964096,avick,fldxt,benjy,whump,grosz,q,avick benjy fldxt grosz whump,5,67043327,0


In [54]:
output_df['word_group'].unique().shape

(538,)